<a href="https://colab.research.google.com/github/yosie111/Shulchan_Aruch_RAG_1/blob/main/SA_RAG_Stage2_Embedding_Benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shulchan Aruch RAG — Stage 2: Embedding + Vector DB + Benchmark

## קלט:
- `shulchan_aruch_v2.jsonl` — chunks מ-Pipeline v2.1
- `על_שוע_חלק_א.xlsx` — 102 שאלות evaluation עם gold citations

## מה המחברת עושה:
1. טוענת chunks + שאלות evaluation
2. מייצרת embeddings עם 3 מודלים (bge-m3, multilingual-e5, NeoDictaBERT)
3. מאחסנת ב-ChromaDB
4. מריצה retrieval benchmark: Recall@k, MRR
5. משווה: vector בלבד / BM25 בלבד / hybrid

## הוראות הרצה:
1. הרץ תא 1 (התקנה — פעם אחת)
2. הרץ תא 2 (טעינת נתונים)
3. הרץ תא 3 (embedding — 2-5 דקות per מודל)
4. הרץ תא 4 (benchmark)
5. הרץ תא 5 (תוצאות + השוואה)


In [4]:
# ============================================================
# תא 1 — התקנה + ייבוא
# ============================================================
!pip install -q sentence-transformers chromadb openpyxl rank-bm25

import json
import re
import os
import time
import numpy as np
from pathlib import Path
from collections import defaultdict

import openpyxl
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# GPU?
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('✅ תא 1 — מוכן')


Device: cuda
  GPU: NVIDIA H100 80GB HBM3
  VRAM: 85.0 GB
✅ תא 1 — מוכן


In [5]:
# ============================================================
# תא 2 — טעינת נתונים: chunks + שאלות evaluation
# ============================================================

# === A. טעינת chunks ===
JSONL_FILE = None
for p in sorted(Path('.').glob('*.jsonl')):
    JSONL_FILE = str(p); break
if JSONL_FILE is None and IN_COLAB:
    print('העלה קובץ JSONL (shulchan_aruch_v2.jsonl):')
    uploaded = colab_files.upload()
    for name in uploaded:
        if name.endswith('.jsonl'): JSONL_FILE = name; break
if JSONL_FILE is None:
    raise FileNotFoundError('לא נמצא קובץ JSONL')

chunks = []
with open(JSONL_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))

# סנן index docs
content_chunks = [c for c in chunks if c.get('metadata', {}).get('doc_type') != 'hilchot_index']
print(f'📄 Chunks: {len(chunks)} total, {len(content_chunks)} content')

# === B. טעינת שאלות evaluation ===
EVAL_FILE = None
for p in sorted(Path('.').glob('*.xlsx')):
    EVAL_FILE = str(p); break
if EVAL_FILE is None and IN_COLAB:
    print('העלה קובץ Excel (שאלות evaluation):')
    uploaded = colab_files.upload()
    for name in uploaded:
        if name.endswith('.xlsx'): EVAL_FILE = name; break
if EVAL_FILE is None:
    raise FileNotFoundError('לא נמצא קובץ Excel')

wb = openpyxl.load_workbook(EVAL_FILE)
ws = wb.active

GEMATRIA = {
    'א':1,'ב':2,'ג':3,'ד':4,'ה':5,'ו':6,'ז':7,'ח':8,'ט':9,
    'י':10,'כ':20,'ל':30,'מ':40,'נ':50,'ס':60,'ע':70,'פ':80,'צ':90,
    'ק':100,'ר':200,'ש':300,'ת':400,
    'ך':20,'ם':40,'ן':50,'ף':80,'ץ':90
}
def heb2num(s):
    return sum(GEMATRIA.get(c,0) for c in re.sub(r'["\'\' ׳״]','',s or ''))

eval_questions = []
for row in range(2, ws.max_row + 1):
    q = ws.cell(row, 1).value
    if not q: continue
    src = ws.cell(row, 3).value or ''
    # חילוץ סימן + סעיף
    siman_match = re.search(r'סימן\s+([א-ת]+)', src)
    seif_match = re.search(r'סעיף\s+([א-ת]+)', src)
    siman_num = heb2num(siman_match.group(1)) if siman_match else 0
    seif_num = heb2num(seif_match.group(1)) if seif_match else 0
    eval_questions.append({
        'question': q,
        'answer': ws.cell(row, 2).value or '',
        'source': src,
        'document': ws.cell(row, 4).value or '',
        'gold_siman': siman_num,
        'gold_seif': seif_num,
    })

print(f'📝 Evaluation: {len(eval_questions)} שאלות')
print(f'   סימנים: {len(set(q["gold_siman"] for q in eval_questions))} ייחודיים')

# === C. מיפוי chunk → gold ===
# כל chunk מכיל level_2_nums (סעיפים). נבדוק אם gold_siman+gold_seif נמצאים בו.
def find_gold_chunk_id(question, chunks):
    """מוצא את ה-chunk_id שמכיל את הסימן+סעיף של השאלה"""
    gs = question['gold_siman']
    gse = question['gold_seif']
    for c in chunks:
        h = c.get('metadata', {}).get('hierarchy', {})
        if h.get('level_1_num') == gs:
            seifs = h.get('level_2_nums', [])
            if gse in seifs or gse == 0:
                return c['doc_id']
    # Fallback: סימן בלבד
    for c in chunks:
        h = c.get('metadata', {}).get('hierarchy', {})
        if h.get('level_1_num') == gs:
            return c['doc_id']
    return None

# Map gold
for q in eval_questions:
    q['gold_chunk_id'] = find_gold_chunk_id(q, content_chunks)

mapped = sum(1 for q in eval_questions if q['gold_chunk_id'])
print(f'   מופו ל-chunk: {mapped}/{len(eval_questions)}')

# דוגמה
ex = eval_questions[0]
print(f'\n   דוגמה: "{ex["question"][:50]}"')
print(f'   gold: סימן {ex["gold_siman"]} סעיף {ex["gold_seif"]} → {ex["gold_chunk_id"]}')

print('\n✅ תא 2 — נתונים טעונים')


העלה קובץ JSONL (shulchan_aruch_v2.jsonl):


Saving shulchan_aruch_v2 (10).jsonl to shulchan_aruch_v2 (10).jsonl
📄 Chunks: 220 total, 214 content
העלה קובץ Excel (שאלות evaluation):


Saving על שוע    חלק א.xlsx to על שוע    חלק א.xlsx
📝 Evaluation: 102 שאלות
   סימנים: 60 ייחודיים
   מופו ל-chunk: 101/102

   דוגמה: "מה צריך האדם לעשות בבוקר ?"
   gold: סימן 1 סעיף 1 → sa_oc_001_1-4__a

✅ תא 2 — נתונים טעונים


In [6]:
# ============================================================
# תא 3 — יצירת Embeddings + אחסון ב-ChromaDB
# ============================================================

# === הגדרת מודלים לבדיקה ===
# הוסף/הסר מודלים לפי הצורך. כל מודל יקבל collection נפרד.
MODELS_TO_TEST = {
    'bge-m3': {
        'model_name': 'BAAI/bge-m3',
        'query_prefix': '',      # bge-m3 לא צריך prefix
        'doc_prefix': '',
    },
    'e5-large': {
        'model_name': 'intfloat/multilingual-e5-large',
        'query_prefix': 'query: ',    # e5 דורש prefix
        'doc_prefix': 'passage: ',
    },
    # הסר הערה אם המודל זמין:
    # 'neodictabert': {
    #     'model_name': 'dicta-il/neodictabert-bilingual',
    #     'query_prefix': '',
    #     'doc_prefix': '',
    # },
}

# === ChromaDB client ===
chroma_client = chromadb.Client()  # in-memory

# === טקסטים מה-chunks ===
chunk_ids = [c['doc_id'] for c in content_chunks]
chunk_texts_emb = [c['content']['text_for_embedding'] for c in content_chunks]
chunk_texts_bm25 = [c['content']['text_for_bm25'] for c in content_chunks]
chunk_texts_llm = [c['content']['text_for_llm'] for c in content_chunks]
chunk_metadatas = [{
    'hilchot_group': c['metadata'].get('general_topic', ''),
    'siman': c['metadata'].get('hierarchy', {}).get('level_1_num', 0),
    'has_rama': c['metadata'].get('has_rama', False),
    'machloket': c['metadata'].get('machloket', False),
} for c in content_chunks]

print(f'Chunks to embed: {len(chunk_ids)}')

# === Embed + store per model ===
model_results = {}  # model_name -> {'model': ..., 'collection': ..., 'time': ...}

for model_key, model_cfg in MODELS_TO_TEST.items():
    print(f'\n{"="*60}')
    print(f'  Model: {model_key} ({model_cfg["model_name"]})')
    print(f'{"="*60}')

    # Load model
    t0 = time.time()
    print(f'  Loading model...')
    model = SentenceTransformer(model_cfg['model_name'], device=DEVICE)
    print(f'  Loaded in {time.time()-t0:.1f}s')

    # Encode documents
    t1 = time.time()
    print(f'  Encoding {len(chunk_texts_emb)} chunks...')
    doc_texts = [model_cfg['doc_prefix'] + t for t in chunk_texts_emb]
    doc_vectors = model.encode(doc_texts, show_progress_bar=True, normalize_embeddings=True)
    encode_time = time.time() - t1
    print(f'  Encoded in {encode_time:.1f}s ({doc_vectors.shape})')

    # Store in ChromaDB
    collection_name = f'sa_{model_key}'.replace('-', '_')
    # Delete if exists
    try: chroma_client.delete_collection(collection_name)
    except: pass
    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={'hnsw:space': 'cosine'}
    )
    collection.add(
        ids=chunk_ids,
        embeddings=[v.tolist() for v in doc_vectors],
        documents=chunk_texts_llm,
        metadatas=chunk_metadatas,
    )
    print(f'  Stored {collection.count()} vectors in ChromaDB')

    model_results[model_key] = {
        'model': model,
        'cfg': model_cfg,
        'collection': collection,
        'encode_time': encode_time,
        'dim': doc_vectors.shape[1],
    }

# === BM25 index ===
print(f'\n{"="*60}')
print(f'  BM25 Index')
print(f'{"="*60}')
tokenized_corpus = [text.split() for text in chunk_texts_bm25]
bm25_index = BM25Okapi(tokenized_corpus)
print(f'  Built BM25 index on {len(tokenized_corpus)} documents')

print(f'\n✅ תא 3 — {len(model_results)} מודלים + BM25 מוכנים')


Chunks to embed: 214

  Model: bge-m3 (BAAI/bge-m3)
  Loading model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

  Loaded in 9.6s
  Encoding 214 chunks...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

  Encoded in 1.1s ((214, 1024))
  Stored 214 vectors in ChromaDB

  BM25 Index
  Built BM25 index on 214 documents

✅ תא 3 — 2 מודלים + BM25 מוכנים


In [7]:
# ============================================================
# תא 4 — Retrieval Benchmark
# ============================================================

K_VALUES = [1, 3, 5, 10]  # Recall@k

def recall_at_k(retrieved_ids, gold_id, k):
    """האם gold_id נמצא ב-top-k?"""
    return 1.0 if gold_id in retrieved_ids[:k] else 0.0

def mrr(retrieved_ids, gold_id):
    """Mean Reciprocal Rank — 1/rank של gold_id"""
    for i, rid in enumerate(retrieved_ids):
        if rid == gold_id:
            return 1.0 / (i + 1)
    return 0.0


def run_vector_benchmark(model_key, model_info, questions, k_max=10):
    """retrieval וקטורי טהור"""
    model = model_info['model']
    cfg = model_info['cfg']
    collection = model_info['collection']

    results = []
    for q in questions:
        if not q['gold_chunk_id']: continue
        query_text = cfg['query_prefix'] + q['question']
        query_vec = model.encode([query_text], normalize_embeddings=True)
        res = collection.query(
            query_embeddings=[query_vec[0].tolist()],
            n_results=k_max,
        )
        retrieved = res['ids'][0]  # list of chunk_ids
        results.append({
            'question': q['question'],
            'gold': q['gold_chunk_id'],
            'retrieved': retrieved,
        })
    return results


def run_bm25_benchmark(questions, k_max=10):
    """retrieval BM25 טהור"""
    results = []
    for q in questions:
        if not q['gold_chunk_id']: continue
        query_tokens = q['question'].split()
        scores = bm25_index.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:k_max]
        retrieved = [chunk_ids[i] for i in top_indices]
        results.append({
            'question': q['question'],
            'gold': q['gold_chunk_id'],
            'retrieved': retrieved,
        })
    return results


def run_hybrid_benchmark(model_key, model_info, questions, k_max=10, vector_weight=0.7):
    """Hybrid: vector + BM25 עם Reciprocal Rank Fusion"""
    model = model_info['model']
    cfg = model_info['cfg']
    collection = model_info['collection']
    rrf_k = 60  # RRF constant

    results = []
    for q in questions:
        if not q['gold_chunk_id']: continue

        # Vector retrieval
        query_text = cfg['query_prefix'] + q['question']
        query_vec = model.encode([query_text], normalize_embeddings=True)
        vec_res = collection.query(
            query_embeddings=[query_vec[0].tolist()],
            n_results=20,
        )
        vec_ids = vec_res['ids'][0]

        # BM25 retrieval
        query_tokens = q['question'].split()
        bm25_scores = bm25_index.get_scores(query_tokens)
        bm25_top = np.argsort(bm25_scores)[::-1][:20]
        bm25_ids = [chunk_ids[i] for i in bm25_top]

        # RRF fusion
        rrf_scores = defaultdict(float)
        for rank, cid in enumerate(vec_ids):
            rrf_scores[cid] += vector_weight / (rrf_k + rank + 1)
        for rank, cid in enumerate(bm25_ids):
            rrf_scores[cid] += (1 - vector_weight) / (rrf_k + rank + 1)

        sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:k_max]
        results.append({
            'question': q['question'],
            'gold': q['gold_chunk_id'],
            'retrieved': sorted_ids,
        })
    return results


def compute_metrics(results, k_values=K_VALUES):
    """חישוב Recall@k ו-MRR"""
    metrics = {}
    n = len(results)
    if n == 0: return {}
    for k in k_values:
        r_at_k = np.mean([recall_at_k(r['retrieved'], r['gold'], k) for r in results])
        metrics[f'Recall@{k}'] = r_at_k
    metrics['MRR'] = np.mean([mrr(r['retrieved'], r['gold']) for r in results])
    metrics['n_questions'] = n
    return metrics


# === Run all benchmarks ===
all_benchmarks = {}

# Filtered questions (only those with gold mapping)
valid_questions = [q for q in eval_questions if q['gold_chunk_id']]
print(f'Running benchmark on {len(valid_questions)} questions...\n')

# BM25
print('Running BM25...')
t0 = time.time()
bm25_results = run_bm25_benchmark(valid_questions)
bm25_metrics = compute_metrics(bm25_results)
all_benchmarks['BM25'] = bm25_metrics
print(f'  Done in {time.time()-t0:.1f}s')
print(f'  Recall@5={bm25_metrics.get("Recall@5",0):.3f}, MRR={bm25_metrics.get("MRR",0):.3f}')

# Vector per model
for model_key, model_info in model_results.items():
    print(f'\nRunning {model_key} (vector)...')
    t0 = time.time()
    vec_results = run_vector_benchmark(model_key, model_info, valid_questions)
    vec_metrics = compute_metrics(vec_results)
    all_benchmarks[f'{model_key}_vector'] = vec_metrics
    print(f'  Done in {time.time()-t0:.1f}s')
    print(f'  Recall@5={vec_metrics.get("Recall@5",0):.3f}, MRR={vec_metrics.get("MRR",0):.3f}')

    # Hybrid
    print(f'Running {model_key} (hybrid)...')
    t0 = time.time()
    hyb_results = run_hybrid_benchmark(model_key, model_info, valid_questions)
    hyb_metrics = compute_metrics(hyb_results)
    all_benchmarks[f'{model_key}_hybrid'] = hyb_metrics
    print(f'  Done in {time.time()-t0:.1f}s')
    print(f'  Recall@5={hyb_metrics.get("Recall@5",0):.3f}, MRR={hyb_metrics.get("MRR",0):.3f}')

print(f'\n✅ תא 4 — benchmark הושלם')


Running benchmark on 101 questions...

Running BM25...
  Done in 0.0s
  Recall@5=0.723, MRR=0.604

Running bge-m3 (vector)...
  Done in 1.3s
  Recall@5=0.881, MRR=0.751
Running bge-m3 (hybrid)...
  Done in 1.1s
  Recall@5=0.871, MRR=0.727

Running e5-large (vector)...
  Done in 1.1s
  Recall@5=0.861, MRR=0.747
Running e5-large (hybrid)...
  Done in 1.2s
  Recall@5=0.871, MRR=0.722

✅ תא 4 — benchmark הושלם


In [8]:
# ============================================================
# תא 5 — תוצאות + השוואה
# ============================================================

print(f'{"="*80}')
print(f'{"RETRIEVAL BENCHMARK RESULTS":^80}')
print(f'{"="*80}')
print()

# Header
header = f'{"Method":<25}'
for k in K_VALUES:
    header += f'{"Recall@"+str(k):>10}'
header += f'{"MRR":>10}'
print(header)
print('-' * len(header))

# Sort by Recall@5
sorted_methods = sorted(all_benchmarks.items(),
                        key=lambda x: x[1].get('Recall@5', 0), reverse=True)

best_recall5 = 0
best_method = ''
for method, metrics in sorted_methods:
    row = f'{method:<25}'
    for k in K_VALUES:
        val = metrics.get(f'Recall@{k}', 0)
        row += f'{val:>10.3f}'
    row += f'{metrics.get("MRR", 0):>10.3f}'
    print(row)
    r5 = metrics.get('Recall@5', 0)
    if r5 > best_recall5:
        best_recall5 = r5
        best_method = method

print(f'\n🏆 Best: {best_method} (Recall@5={best_recall5:.3f})')

# === Error Analysis: שאלות שנכשלו ===
print(f'\n{"="*80}')
print(f'{"ERROR ANALYSIS — Failed at Recall@5":^80}')
print(f'{"="*80}')

# Use best method for error analysis
best_key = best_method.replace('_hybrid', '').replace('_vector', '')
if best_key in model_results:
    if '_hybrid' in best_method:
        analysis_results = run_hybrid_benchmark(best_key, model_results[best_key], valid_questions)
    else:
        analysis_results = run_vector_benchmark(best_key, model_results[best_key], valid_questions)
else:
    analysis_results = bm25_results

failed = []
for r in analysis_results:
    if r['gold'] not in r['retrieved'][:5]:
        failed.append(r)

print(f'\nFailed: {len(failed)}/{len(analysis_results)}')
print()
for f in failed[:10]:
    q_text = f['question'][:60]
    gold = f['gold']
    top3 = f['retrieved'][:3]
    print(f'  Q: {q_text}')
    print(f'  Gold: {gold}')
    print(f'  Got:  {top3}')
    print()

# === Model Speed Comparison ===
print(f'{"="*80}')
print(f'{"MODEL INFO":^80}')
print(f'{"="*80}')
for mk, mi in model_results.items():
    print(f'  {mk}: dim={mi["dim"]}, encode_time={mi["encode_time"]:.1f}s')

# === Hybrid vs Vector improvement ===
print(f'\n{"="*80}')
print(f'{"HYBRID IMPROVEMENT":^80}')
print(f'{"="*80}')
for mk in model_results:
    vec_r5 = all_benchmarks.get(f'{mk}_vector', {}).get('Recall@5', 0)
    hyb_r5 = all_benchmarks.get(f'{mk}_hybrid', {}).get('Recall@5', 0)
    bm25_r5 = all_benchmarks.get('BM25', {}).get('Recall@5', 0)
    delta = hyb_r5 - vec_r5
    print(f'  {mk}: vector={vec_r5:.3f} → hybrid={hyb_r5:.3f} (Δ={delta:+.3f})')
print(f'  BM25 alone: {bm25_r5:.3f}')

print(f'\n✅ תא 5 — סיום')


                          RETRIEVAL BENCHMARK RESULTS                           

Method                     Recall@1  Recall@3  Recall@5 Recall@10       MRR
---------------------------------------------------------------------------
bge-m3_vector                 0.673     0.812     0.881     0.931     0.751
bge-m3_hybrid                 0.624     0.812     0.871     0.950     0.727
e5-large_hybrid               0.604     0.822     0.871     0.921     0.722
e5-large_vector               0.663     0.812     0.861     0.921     0.747
BM25                          0.515     0.673     0.723     0.802     0.604

🏆 Best: bge-m3_vector (Recall@5=0.881)

                      ERROR ANALYSIS — Failed at Recall@5                       

Failed: 12/101

  Q: האם מותר לאחר את זמן התפילה ?
  Gold: sa_oc_001_1-4__a
  Got:  ['sa_oc_089_1-2__a', 'sa_oc_090_15-18__d', 'sa_oc_093_1-4']

  Q: כמה משמרות יש בלילה ?
  Gold: sa_oc_001_1-4__a
  Got:  ['sa_oc_018_1-3', 'sa_oc_058_4-7__b', 'sa_oc_047_12-14__b'